In [55]:
# Imports Libraries
import random
import heapq
from copy import deepcopy

In [56]:
# Global Configuration
StateDimension = 4  # Change this to 3 or 4 to switch between 8-puzzle and 15-puzzle
GoalState = list(range(1, StateDimension**2)) + [0]

Opposite = {'u': 'd', 'd': 'u', 'l': 'r', 'r': 'l'}

In [57]:
# Action functions
def Actions(state):
    return ['u', 'd', 'l', 'r']

def Result(state, action):
    i = state.index(0)
    new_state = list(state)
    row, col = divmod(i, StateDimension)
    target = None
    if action == 'u' and row > 0:
        target = i - StateDimension
    elif action == 'd' and row < StateDimension - 1:
        target = i + StateDimension
    elif action == 'l' and col > 0:
        target = i - 1
    elif action == 'r' and col < StateDimension - 1:
        target = i + 1
    if target is not None:
        new_state[i], new_state[target] = new_state[target], new_state[i]
        return new_state
    return state


def GoalTest(state):
    return state == GoalState

In [58]:
# Heuristics
def H_OutOfPlace(state):
    return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != GoalState[i])


def H_Manhattan(state):
    total = 0
    for i, tile in enumerate(state):
        if tile == 0:
            continue
        goal_i = GoalState.index(tile)
        r1, c1 = divmod(i, StateDimension)
        r2, c2 = divmod(goal_i, StateDimension)
        total += abs(r1 - r2) + abs(c1 - c2)
    return total

In [59]:
# Search node class
class Node:
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.heuristic = heuristic

    def __lt__(self, other):
        return (self.cost + self.heuristic) < (other.cost + other.heuristic)

    def path(self):
        node, p = self, []
        while node:
            p.append(node)
            node = node.parent
        return list(reversed(p))

In [60]:
# BFS search
def BFS(start):
    frontier = [Node(start)]
    explored = set()
    nodes_expanded = 0
    while frontier:
        node = frontier.pop(0)
        if tuple(node.state) in explored:
            continue
        explored.add(tuple(node.state))
        nodes_expanded += 1
        if GoalTest(node.state):
            return node.path(), nodes_expanded
        for a in Actions(node.state):
            child_state = Result(node.state, a)
            if tuple(child_state) not in explored:
                frontier.append(Node(child_state, node, a))
    return None, nodes_expanded

In [61]:
# A* search
def A_star(start, heuristic_func):
    frontier = [Node(start, heuristic=heuristic_func(start))]
    explored = {}
    nodes_expanded = 0
    while frontier:
        node = heapq.heappop(frontier)
        if GoalTest(node.state):
            return node.path(), nodes_expanded
        key = tuple(node.state)
        if key in explored and explored[key] <= node.cost:
            continue
        explored[key] = node.cost
        nodes_expanded += 1
        for a in Actions(node.state):
            child_state = Result(node.state, a)
            if child_state == node.state:
                continue
            child = Node(child_state, node, a, node.cost + 1, heuristic_func(child_state))
            heapq.heappush(frontier, child)
    return None, nodes_expanded

In [62]:
# Generate random puzzle
def random_walk(steps):
    state = GoalState[:]
    last_action = None
    for _ in range(steps):
        options = [a for a in Actions(state) if a != Opposite.get(last_action)]
        action = random.choice(options)
        state = Result(state, action)
        last_action = action
    return state

In [63]:
# Generate and test puzzles
def run_experiments(dim):
    global StateDimension, GoalState
    StateDimension = dim
    GoalState = list(range(1, StateDimension ** 2)) + [0]

    step_counts = [5, 10, 20, 40, 80]
    problems = []
    for steps in step_counts:
        for _ in range(3):
            problems.append(random_walk(steps))

    results = []
    for idx, problem in enumerate(problems):
        entry = {'start_state': problem}

        # BFS (skip for 4x4 if >10 steps)
        if StateDimension == 3 or step_counts[idx // 3] <= 10:
            path, expanded = BFS(problem)
            entry['BFS'] = {
                'solution': [node.action for node in path][1:] if path else None,
                'length': len(path) - 1 if path else None,
                'expanded': expanded
            }

        # A* Out of Place
        path, expanded = A_star(problem, H_OutOfPlace)
        entry['A*_OutOfPlace'] = {
            'solution': [node.action for node in path][1:] if path else None,
            'length': len(path) - 1 if path else None,
            'expanded': expanded
        }

        # A* Manhattan
        path, expanded = A_star(problem, H_Manhattan)
        entry['A*_Manhattan'] = {
            'solution': [node.action for node in path][1:] if path else None,
            'length': len(path) - 1 if path else None,
            'expanded': expanded
        }

        results.append(entry)
    return results

In [66]:
# Run 3x3 experiments
results_3x3 = run_experiments(3)

(results_3x3) [:15]

[{'start_state': [1, 2, 3, 4, 5, 6, 7, 8, 0],
  'BFS': {'solution': [], 'length': 0, 'expanded': 1},
  'A*_OutOfPlace': {'solution': [], 'length': 0, 'expanded': 0},
  'A*_Manhattan': {'solution': [], 'length': 0, 'expanded': 0}},
 {'start_state': [1, 5, 2, 4, 0, 3, 7, 8, 6],
  'BFS': {'solution': ['u', 'r', 'd', 'd'], 'length': 4, 'expanded': 24},
  'A*_OutOfPlace': {'solution': ['u', 'r', 'd', 'd'],
   'length': 4,
   'expanded': 4},
  'A*_Manhattan': {'solution': ['u', 'r', 'd', 'd'],
   'length': 4,
   'expanded': 4}},
 {'start_state': [1, 2, 0, 4, 5, 3, 7, 8, 6],
  'BFS': {'solution': ['d', 'd'], 'length': 2, 'expanded': 4},
  'A*_OutOfPlace': {'solution': ['d', 'd'], 'length': 2, 'expanded': 2},
  'A*_Manhattan': {'solution': ['d', 'd'], 'length': 2, 'expanded': 2}},
 {'start_state': [4, 1, 3, 7, 2, 6, 5, 0, 8],
  'BFS': {'solution': ['l', 'u', 'u', 'r', 'd', 'd', 'r'],
   'length': 7,
   'expanded': 149},
  'A*_OutOfPlace': {'solution': ['l', 'u', 'u', 'r', 'd', 'd', 'r'],
   'l

In [65]:
# Run 4x4 experiments
results_4x4 = run_experiments(4)

(results_4x4)[:1]

[{'start_state': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 11, 13, 14, 12, 0],
  'BFS': {'solution': ['l', 'u', 'r', 'd'], 'length': 4, 'expanded': 37},
  'A*_OutOfPlace': {'solution': ['l', 'u', 'r', 'd'],
   'length': 4,
   'expanded': 5},
  'A*_Manhattan': {'solution': ['l', 'u', 'r', 'd'],
   'length': 4,
   'expanded': 4}}]

While search is an essential function in AI for things like puzzle problem solving, planning, and decision-making. As we see in the experiment above when the puzzle size is increased form 3x3 to 4x4 the number of possible states grow exponentially, making simple search methods like BFS inefficient.